In [3]:
import requests
import pandas as pd
import time
import json
import os
from datetime import datetime
from tqdm.auto import tqdm
from openai import OpenAI

In [ ]:
# Clean display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [4]:
# Initialize OpenAI
client = OpenAI(api_key="YOUR_OPENAI_API_KEY")

In [5]:
def get_reddit_leads(subreddits, keywords, hours_limit=24):
    """Scrapes Reddit safely and pre-filters using keywords to save API costs."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}
    current_time = time.time()
    leads = []
    keywords = [k.lower() for k in keywords]

    for sub in tqdm(subreddits, desc="Scraping Subreddits"):
        url = f"https://www.reddit.com/r/{sub}/new.json?limit=100"
        try:
            response = requests.get(url, headers=headers)
            if response.status_code == 429:
                print(f"Rate limited on r/{sub}. Skipping.")
                continue
            elif response.status_code != 200:
                continue 
                
            posts = response.json().get('data', {}).get('children', [])
            
            for post in posts:
                d = post['data']
                post_time = d.get('created_utc', 0)
                hours_ago = (current_time - post_time) / 3600
                
                if hours_ago > hours_limit:
                    continue
                    
                title = d.get('title', '')
                text = d.get('selftext', '')
                combined_text = f"{title} {text}".lower()
                
                # PRE-FILTER: Only keep it if it matches a keyword
                if any(k in combined_text for k in keywords):
                    leads.append({
                        "Post_ID": d.get('id'), # Unique ID for our database
                        "Subreddit": d.get('subreddit'),
                        "Hours_Ago": round(hours_ago, 1),
                        "Title": title.strip(),
                        "Full_Text": text.strip().replace('\n', ' '),
                        "Post_URL": f"https://reddit.com{d.get('permalink')}"
                    })
        except Exception:
            pass 
        time.sleep(1.5) # Safety delay

    df = pd.DataFrame(leads)
    return df

In [6]:
def grade_leads_with_llm(df_new_leads, db_path="graded_leads_db.csv", min_score=80):
    """Grades leads via LLM, but checks local CSV first to prevent duplicate API costs."""
    if df_new_leads.empty:
        print("No new leads to grade.")
        return pd.DataFrame()

    # Load existing database to avoid re-grading
    if os.path.exists(db_path):
        df_db = pd.read_csv(db_path)
        graded_ids = df_db['Post_ID'].tolist()
    else:
        df_db = pd.DataFrame()
        graded_ids = []

    new_results = []
    
    for _, row in tqdm(df_new_leads.iterrows(), total=df_new_leads.shape[0], desc="AI Grading"):
        # Cost Saver: Skip if already graded
        if row['Post_ID'] in graded_ids:
            continue
            
        post_text = f"Title: {row['Title']}\nText: {row['Full_Text']}"
        prompt = """
        You evaluate leads for 'QRDive', a tool that catches bad reviews privately and pushes 5-star reviews to Google Maps.
        Rate this Reddit post from 0-100 on how badly they need this tool.
        - 90-100: Actively complaining about bad reviews, or wants more 5-star reviews.
        - 0-89: General business talk, or already has a solution.
        Respond ONLY in JSON format: {"score": 95, "reason": "Short explanation"}
        """

        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                response_format={ "type": "json_object" },
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": post_text}
                ],
                temperature=0.1
            )
            
            result = json.loads(response.choices[0].message.content)
            
            row_dict = row.to_dict()
            row_dict['AI_Score'] = result.get('score', 0)
            row_dict['AI_Reason'] = result.get('reason', '')
            new_results.append(row_dict)
            
        except Exception as e:
            print(f"Error grading {row['Post_ID']}")

    # Save new results to database to prevent future re-grading
    if new_results:
        df_new_graded = pd.DataFrame(new_results)
        df_db = pd.concat([df_db, df_new_graded], ignore_index=True)
        df_db.to_csv(db_path, index=False)
    
    # Filter and return only the high-quality leads
    if not df_db.empty:
        df_high_quality = df_db[df_db['AI_Score'] >= min_score].copy()
        df_high_quality = df_high_quality.sort_values(by='AI_Score', ascending=False).reset_index(drop=True)
        return df_high_quality
    return pd.DataFrame()

In [7]:
# 1. Targets
my_subs = [
    "smallbusiness", "startups", "RestaurantOwners", "barber", 
    "digital_marketing", "Spa_and_Salon_Owners", "Entrepreneur", "sweatystartup"
]

my_keys = [
    "bad review", "google review", "yelp", "1-star", "fake review", 
    "review bombed", "unfair review", "get more reviews", "customer feedback"
]

In [8]:
# 2. Scrape (Pre-filter)
print("Step 1: Scraping for raw keyword matches...")
df_raw = get_reddit_leads(subreddits=my_subs, keywords=my_keys, hours_limit=48)
print(f"Found {len(df_raw)} raw matches.")

Step 1: Scraping for raw keyword matches...


Scraping Subreddits: 100%|██████████| 8/8 [00:22<00:00,  2.86s/it]

Found 1 raw matches.


In [10]:
df_raw

,Post_ID,Subreddit,Hours_Ago,Title,Full_Text,Post_URL
0,1ryeosm,smallbusiness,2.1,3 years running a junk removal business solo. ...,been running a junk removal company in LA for ...,https://reddit.com/r/smallbusiness/comments/1r...


In [ ]:
# 3. Grade (LLM + Database)
print("\nStep 2: AI Grading...")
# It will automatically skip posts it has seen before and save new ones to "graded_leads_db.csv"
df_final = grade_leads_with_llm(df_new_leads=df_raw, min_score=85)

In [ ]:
# 4. View Results
print("\n--- 🔥 TOP LEADS TODAY 🔥 ---")
if not df_final.empty:
    display(df_final[['AI_Score', 'AI_Reason', 'Subreddit', 'Title', 'Post_URL']])
else:
    print("No high-quality leads found today.")